In [1]:
import sys
import os

if not os.path.exists("config.py"):
    os.chdir("backend") if os.path.exists("backend") else os.chdir("..")

sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())
print("config.py exists:", os.path.exists("config.py"))

Working directory: c:\Users\Dell\Documents\repo\llms\document-assistant\backend
config.py exists: True


In [2]:
from openai import OpenAI
from supabase import create_client
from config import settings

openai_client = OpenAI(api_key=settings.openai_api_key)
sb = create_client(settings.supabase_url, settings.supabase_service_role_key)

print("Clients ready")

Clients ready


In [3]:
def search_chunks(query: str, top_k: int = 5, project: str = None) -> list[dict]:
    # Embed the question using the same model as the chunks
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )
    query_embedding = response.data[0].embedding

    # Call the match_chunks function we created in Supabase
    result = sb.rpc("match_chunks", {
        "query_embedding": query_embedding,
        "match_count": top_k,
        "filter_project": project
    }).execute()

    return result.data

# Test question
results = search_chunks("What are the penalties for late completion?")

for r in results:
    print(f"Clause: {r['clause_ref']} — Similarity: {r['similarity']:.4f}")
    print(r['content'][:200])
    print()

Clause: 17.4.1 — Similarity: 0.6224
17.4.1 In the event that the Contractor fails to complete the Works, a Section or Key Stage, as
the case may be, within the Time for Completion applicable thereto, the Contractor shall
pay to the Auth

Clause: 17.6.1 — Similarity: 0.5126
17.6.1 In the event that the Contractor fails to comply with its obligations pursuant to in Sub-
clause 3.20.1 or Sub-clause 3.20.2, and notwithstanding any later date fixed by the
Engineer for compli

Clause: 3.20.4 — Similarity: 0.5069
3.20.4 In the event that the Contractor fails to achieve compliance by the date fixed by the
Engineer pursuant to Sub-clause 3.20.3:
(a) the Contractor shall pay to the Authority the Penalties require

Clause: 17.5.1 — Similarity: 0.5042
17.5.1 In the event that the Contractor fails to comply with its obligations pursuant to in Sub-
clause 3.19.1 or Sub-clause 3.19.2, and notwithstanding any later date fixed by the
Engineer for compli

Clause: 17.7.1 — Similarity: 0.5011
17.7.1 In th

In [4]:
def build_prompt(query: str, chunks: list[dict]) -> str:
    context = ""
    for chunk in chunks:
        context += f"Clause {chunk['clause_ref']}:\n{chunk['content']}\n\n"

    prompt = f"""You are a contract assistant helping a Quantity Surveyor analyse contract documents.
Answer the question using only the clauses provided below. 
Always cite the clause reference in your answer.
If the answer is not found in the clauses, say so clearly.

---
{context}
---

Question: {query}

Answer:"""
    return prompt

# Test it
prompt = build_prompt("What are the penalties for late completion?", results)
print(prompt[:1500])

You are a contract assistant helping a Quantity Surveyor analyse contract documents.
Answer the question using only the clauses provided below. 
Always cite the clause reference in your answer.
If the answer is not found in the clauses, say so clearly.

---
Clause 17.4.1:
17.4.1 In the event that the Contractor fails to complete the Works, a Section or Key Stage, as
the case may be, within the Time for Completion applicable thereto, the Contractor shall
pay to the Authority Penalties at the rate stated in section 7-A [Penalties] of Appendix 2
[Contract Particulars] as Penalties for delay for each Day or part Day which shall elapse:
(a) in the case of a failure to complete a Section, between the Time for Completion
for that Section and the Taking-Over Date for that Section; and
(b) in the case of a failure to complete the Works, between the Time for Completion
for the Works and the Completion Date; and
(c) in the case of a failure to achieve a Key Stage, between the Time for Completion


In [5]:
def ask(query: str, project: str = None, top_k: int = 5) -> dict:
    # Step 1: retrieve relevant chunks
    chunks = search_chunks(query, top_k=top_k, project=project)
    
    # Step 2: build prompt
    prompt = build_prompt(query, chunks)
    
    # Step 3: send to GPT
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    answer = response.choices[0].message.content
    sources = [{"clause_ref": c["clause_ref"], "similarity": c["similarity"]} for c in chunks]

    return {
        "answer": answer,
        "sources": sources
    }

# Test it
result = ask("What are the penalties for late completion?")
print(result["answer"])
print("\n--- Sources ---")
for s in result["sources"]:
    print(f"  Clause {s['clause_ref']} (similarity: {s['similarity']:.4f})")

The penalties for late completion are specified in Clause 17.4.1. If the Contractor fails to complete the Works, a Section, or a Key Stage within the applicable Time for Completion, the Contractor shall pay Penalties at the rate stated in section 7-A [Penalties] of Appendix 2 [Contract Particulars] for each Day or part Day that elapses as follows:

(a) For a failure to complete a Section, between the Time for Completion for that Section and the Taking-Over Date for that Section;
(b) For a failure to complete the Works, between the Time for Completion for the Works and the Completion Date; 
(c) For a failure to achieve a Key Stage, between the Time for Completion for that Key Stage and the date on which the Engineer gives its non-objection to the completion requirements for that Key Stage as stated in the Project Brief. 

(Refer to Clause 17.4.1)

--- Sources ---
  Clause 17.4.1 (similarity: 0.6224)
  Clause 17.6.1 (similarity: 0.5126)
  Clause 3.20.4 (similarity: 0.5069)
  Clause 17.5.

In [6]:
result = ask("What are the requirements for the performance guarantee?")
print(result["answer"])
print("\n--- Sources ---")
for s in result["sources"]:
    print(f"  Clause {s['clause_ref']} (similarity: {s['similarity']:.4f})")

The requirements for the performance guarantee are outlined in the following clauses:

1. The Contractor must provide a Performance Guarantee to the Authority as per Sub-clause 1.11 [Performance Guarantee] (Clause 1.1.98).
2. The Performance Guarantee must remain valid until ninety (90) days after the issuance of the Maintenance Certificate (Clause 1.11.5).
3. If the Contractor has not received the Maintenance Certificate by seventy (70) days prior to the expiry of the Performance Guarantee, the Engineer will notify the Contractor to extend it at their own expense by an additional thirty (30) days, unless a different period is specified (Clause 1.11.5).
4. If the Contractor fails to extend, replace, or amend the Performance Guarantee prior to thirty-five (35) days of its expiry, the Authority may demand payment under the Performance Guarantee (Clause 1.11.5).
5. If the Contractor fails to deliver a new or amended Performance Guarantee within fourteen (14) days after the Engineer’s noti

In [3]:
result = sb.table("contract_chunks") \
    .select("clause_ref, content") \
    .eq("filename", "Scope of Works.pdf") \
    .limit(10) \
    .execute()

for row in result.data:
    print(f"Clause: {row['clause_ref']}")
    print(row['content'][:200])
    print()

Clause: 3.11
3.11 Site Investigation and Geotechnical Works (If Applicable)

Clause: 3.15
3.15 Geotechnical Investigation Report (If Applicable)

Clause: 3.19
3.19 Programmes
4 AUTHORITY NOMINATED SUB-CONTRACTOR
BP 2023 C 043 S Schedule A: Part 1: Scope of Works / Contents Rev. 0
Design and Build for the Renovation of the Heritage Buildings of the Qatar Med

Clause: 1.1.1
1.1.1 This document is the first part of (7) documents which will comprise SCHEDULE (A)
– Project Brief, referred to within the Building and Engineering Works General
Conditions of Contract for Design

Clause: 1.1.2
1.1.2 All Appendices to the Project Brief are to be read and construed as a composite whole
and shall be taken as mutually explanatory of one another. In the event of an
ambiguity, discrepancy or inco

Clause: 1.1.3
1.1.3 This document sets out the duties to be performed by the Contractor and the
Deliverables required from the Contractor.

Clause: 1.2.1
1.2.1 The Contractor shall be appointed under the ter

In [4]:
result = sb.table("contract_chunks") \
    .select("clause_ref, content") \
    .eq("filename", "Scope of Works.pdf") \
    .ilike("content", "%dismantl%") \
    .execute()

for row in result.data:
    print(f"Clause: {row['clause_ref']}")
    print(row['content'][:500])
    print()

Clause: 3.2.2 (part 4)
that has cracked; concrete that is honeycombed, fractured, excessive surface depressions, or otherwise defective must be removed and reinstated the surface/element to its original shape and strength.  When deemed necessary and instructed by the Engineer or Engineer's representative, a cathodic protection system for the reinforcement could be introduced to control the corrosion in the reinforcement bars.  Establish safety protocols and measures to ensure the well-being of workers and the surrou

Clause: 3.2.2 (part 5)
250Kva: 1 unit (battery autonomy 30 minutes at full load) at the Old TV Building - 20Kva: 1 unit (battery autonomy 30 minutes at full load), at the Old TV Building, ensuring uninterrupted power supply for critical systems.  Rewire electrical systems to comply with Kahramaa standards, ensuring safety and adherence to electrical codes.  All existing main cable sizes must be adhered and verified to Kahramaa regulations and must comply with the allow

In [5]:
result = sb.table("ingested_files") \
    .select("filename, chunk_count") \
    .execute()

for row in result.data:
    print(f"{row['filename']}: {row['chunk_count']} chunks")

Scope of Works.pdf: 122 chunks
Contract C2024-49.pdf: 773 chunks
